In [ ]:
# KALMAN OPEN POLICY SOURCE-OF-TRUTH RECOVERY v2.9 — ONE CELL / READ ONLY
# Searches Drive text artifacts + local mounted Drive for the original OPEN_* policy specification/code.
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
from pathlib import Path
import os,re,json,subprocess,sys

TOKENS=[
"OPEN_NEG_0BP_0M","OPEN_NEG_20BP_0M","OPEN_FLIP_0M","OPEN_NEG_5M_CONFIRM",
"OPEN_FLIP_5M_CONFIRM","OPEN_GAP_5M_CONFIRM","OPEN_GIVEBACK_5M","OPEN_NEG_15M_CONFIRM"
]
roots=[
Path("/content/drive/MyDrive/US_ETF"),
Path("/content/drive/MyDrive")
]
exts={".py",".ipynb",".md",".txt",".json",".yaml",".yml",".csv",".tsv",".log"}
hits=[]
seen=set()
print("[SEARCH TOKENS]",TOKENS)
for root in roots:
    if not root.exists(): continue
    for p in root.rglob("*"):
        try:
            if not p.is_file() or p.suffix.lower() not in exts: continue
            sp=str(p)
            if sp in seen: continue
            seen.add(sp)
            if p.stat().st_size>20_000_000: continue
            txt=p.read_text(errors="ignore")
            found=[t for t in TOKENS if t in txt]
            if found:
                hits.append((sp,found,txt))
        except Exception:
            pass
print("[FILES WITH POLICY TOKENS]",len(hits))
for sp,found,txt in hits:
    print("\nFILE",sp,"TOKENS",found)
    lines=txt.splitlines()
    idx=[i for i,l in enumerate(lines) if any(t in l for t in found)]
    for i in idx[:30]:
        lo=max(0,i-8); hi=min(len(lines),i+15)
        print(f"--- context L{lo+1}-{hi} ---")
        print("\n".join(lines[lo:hi]))

# Also search semantic fragments likely used before names were assigned.
patterns=[
r"position_return_open",r"position_return_5m",r"position_return_15m",
r"overnight_gap_return",r"giveback_prev_close_to_5m",
r"NEG_20BP",r"5M_CONFIRM",r"15M_CONFIRM",r"open[_ ]?revalidation"
]
sem=[]
for root in roots[:1]:
    if not root.exists(): continue
    for p in root.rglob("*"):
        try:
            if not p.is_file() or p.suffix.lower() not in exts or p.stat().st_size>20_000_000: continue
            txt=p.read_text(errors="ignore")
            score=sum(bool(re.search(pt,txt,re.I)) for pt in patterns)
            if score>=3 and not any(str(p)==x[0] for x in hits):
                sem.append((score,str(p),txt))
        except: pass
sem=sorted(sem,reverse=True)[:30]
print("\n[SEMANTIC CANDIDATES]",len(sem))
for score,sp,txt in sem:
    print("\nFILE",sp,"SCORE",score)
    lines=txt.splitlines()
    ids=[i for i,l in enumerate(lines) if any(re.search(pt,l,re.I) for pt in patterns)]
    for i in ids[:12]:
        print(f"L{i+1}: {lines[i][:1000]}")

print("\n[DECISION]")
if hits:
    print("Exact policy-token artifacts found. Inspect contexts above and recover formulas from source, not from Gold-20 fitting.")
else:
    print("No exact-token source found in mounted Drive text artifacts.")
    print("Use Git history / older notebooks / generated checklist provenance next; do not infer ambiguous policies from Gold-20.")
print("READ ONLY. No canonical file modified.")
